In [19]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas
hv.extension('bokeh')

from pprint import pprint
from scipy import stats

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import the class
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session
from matplotlib import pyplot as plt

print("✓ Imports loaded successfully!")

✓ Imports loaded successfully!


## 1. Load cell via session

In [20]:
# Load MSN cell database
monkey = 'fiona'  # 'yasmin' or 'fiona'

base_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' 
pickle_file = base_path / f'msn_{monkey}_cell_trial_data.pkl'

cell_df = pd.read_pickle(pickle_file)

# Select a session to analyze - use session with most cells
session_cell_counts = cell_df.groupby('trial_session')['cell_ID'].nunique().sort_values(ascending=False)
best_session_id = session_cell_counts.index[0]
best_session_id = "fi210824a"

print(f"Using session with most cells: {best_session_id}")
print(f"Number of cells in this session: {session_cell_counts.iloc[0]}")

# Get all data for this session
session_data = cell_df[cell_df['trial_session'] == best_session_id]

# Create Session object
session = Session(session_data, verbose=True)

Using session with most cells: fi210824a
Number of cells in this session: 85
Session fi210824a initialized:
  - Number of cells: 6
  - Total trials: 546
  - Trial types: ['CONT', 'GO', 'STOP']
  - Directions: [np.int64(0), np.int64(180)]


In [21]:
i = 16
epok = [-500, 500]
align = "go_cue"
type = 'GO'

In [22]:
# Get a sample cell (e.g., the 10th cell)
# i = 9
# cell = session.get_cell(session.cell_ids[i])
cell = session.get_cell(9876)
print(f"Analyzing Cell ID: {cell.cell_id}")

Analyzing Cell ID: 9876


## 2. ANOVA Classification

In [ ]:
def perform_cell_anova(cell):
    """
    Perform ANOVA tests to determine cell sensitivity to various conditions.
    
    Conditions tested:
    1. Task Modulation: Baseline vs. GO Phase (All directions)
    2. GO Directionality: GO Left vs. GO Right (during GO phase)
    3. STOP Directionality: STOP Left vs. STOP Right (during STOP phase)
    4. CONT Directionality: CONT Left vs. CONT Right (during CONT phase)
    5. Signal Sensitivity: STOP vs. CONT (aligned to STOP cue)
    6. Right Modulation: Baseline vs. GO Right vs. STOP Right
    7. Left Modulation: Baseline vs. GO Left vs. STOP Left
    """
    
    results = {}
    
    # Helper to extract firing rates manually
    def get_rates(align_event, window, trial_type=None, direction=None, success_only=True):
        # Filter trials
        df = cell.filter_trials(trial_type=trial_type, direction=direction, success_only=success_only)
        
        rates = []
        for _, row in df.iterrows():
            # Get alignment time
            t0 = np.nan
            
            if align_event == 'go_cue':
                t0 = row['go_cue']
            elif align_event == 'stop_cue':
                # Both STOP and CONT trials have a stop_cue
                t0 = row['stop_cue']
            
            # Skip if alignment event is missing
            if pd.isna(t0):
                continue
                
            spikes = np.array(row['neural_data'])
            # Align spikes
            aligned_spikes = spikes - t0
            
            # Count spikes in window
            count = np.sum((aligned_spikes >= window[0]) & (aligned_spikes <= window[1]))
            duration = (window[1] - window[0]) / 1000.0
            rates.append(count / duration)
            
        return np.array(rates)

    # 1. Task Modulation (Baseline vs GO)
    # Baseline: [-500, 0] aligned to GO
    # GO: [0, 500] aligned to GO
    base_rates = get_rates('go_cue', [-500, 0], trial_type='GO')
    go_rates = get_rates('go_cue', [0, 500], trial_type='GO')
    
    if len(base_rates) > 0 and len(go_rates) > 0:
        f_stat, p_val = stats.f_oneway(base_rates, go_rates)
        results['Task_Modulation'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Task_Modulation'] = {'F': np.nan, 'p': np.nan, 'significant': False}
    
    # 2. GO Directionality
    # GO Left (180) vs GO Right (0) in [0, 500] aligned to GO
    go_left = get_rates('go_cue', [0, 500], trial_type='GO', direction=180)
    go_right = get_rates('go_cue', [0, 500], trial_type='GO', direction=0)
    
    if len(go_left) > 0 and len(go_right) > 0:
        f_stat, p_val = stats.f_oneway(go_left, go_right)
        results['GO_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['GO_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 3. STOP Directionality
    # STOP Left vs STOP Right in [0, 200] aligned to STOP cue
    stop_left = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=180)
    stop_right = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=0)
    
    if len(stop_left) > 0 and len(stop_right) > 0:
        f_stat, p_val = stats.f_oneway(stop_left, stop_right)
        results['STOP_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['STOP_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 4. CONT Directionality
    # CONT Left vs CONT Right in [0, 500] aligned to GO (since no stop cue)
    # Using same window as GO
    cont_left = get_rates('go_cue', [0, 500], trial_type='CONT', direction=180)
    cont_right = get_rates('go_cue', [0, 500], trial_type='CONT', direction=0)
    
    if len(cont_left) > 0 and len(cont_right) > 0:
        f_stat, p_val = stats.f_oneway(cont_left, cont_right)
        results['CONT_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['CONT_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 5. Signal Sensitivity (STOP vs CONT)
    # Compare STOP vs CONT aligned to STOP cue
    # Window: [0, 200] ms after STOP cue
    stop_rates_sig = get_rates('stop_cue', [0, 200], trial_type='STOP')
    cont_rates_sig = get_rates('stop_cue', [0, 200], trial_type='CONT')
    
    if len(stop_rates_sig) > 0 and len(cont_rates_sig) > 0:
        f_stat, p_val = stats.f_oneway(stop_rates_sig, cont_rates_sig)
        results['Signal_Sensitivity'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Signal_Sensitivity'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 6. Right Modulation (Baseline vs GO Right vs STOP Right)
    # Baseline: [-500, 0] aligned to GO (from GO trials)
    # GO Right: [0, 500] aligned to GO (from GO trials, dir 0)
    # STOP Right: [0, 200] aligned to STOP (from STOP trials, dir 0)
    base_rates_right = get_rates('go_cue', [-500, 0], trial_type='GO') # Baseline is general
    go_right_rates = get_rates('go_cue', [0, 500], trial_type='GO', direction=0)
    stop_right_rates = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=0)

    if len(base_rates_right) > 0 and len(go_right_rates) > 0 and len(stop_right_rates) > 0:
        f_stat, p_val = stats.f_oneway(base_rates_right, go_right_rates, stop_right_rates)
        results['Right_Modulation'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Right_Modulation'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 7. Left Modulation (Baseline vs GO Left vs STOP Left)
    # Baseline: [-500, 0] aligned to GO (from GO trials)
    # GO Left: [0, 500] aligned to GO (from GO trials, dir 180)
    # STOP Left: [0, 200] aligned to STOP (from STOP trials, dir 180)
    base_rates_left = get_rates('go_cue', [-500, 0], trial_type='GO') # Baseline is general
    go_left_rates = get_rates('go_cue', [0, 500], trial_type='GO', direction=180)
    stop_left_rates = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=180)

    if len(base_rates_left) > 0 and len(go_left_rates) > 0 and len(stop_left_rates) > 0:
        f_stat, p_val = stats.f_oneway(base_rates_left, go_left_rates, stop_left_rates)
        results['Left_Modulation'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Left_Modulation'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    return results

# Run analysis
anova_results = perform_cell_anova(cell)
print(f"ANOVA Results for Cell {cell.cell_id}:")
pprint(anova_results)

GO Left rates: [ 8.  6.  4.  2.  2.  4.  4.  4.  4.  0.  6.  0.  4.  4.  4.  4.  2.  2.
  0.  0.  2.  6.  0. 12.  6.  0.  4.  2. 12.  0.  4. 10.  4.  2.  0.  0.
  4.  2.  2. 12.  4.  2.  4.  2.  4.  2.  6.  0.  0.  0. 14.  0.  6.  2.
  4.  4.  6.  2.  0.  2.  2.  2.  8. 12.  2.  4.  2.  8.  4.  4. 10.  2.
  2. 12.  2.  0.  2.  2.  0.  0.  4.  6.  2. 10.  2.  4.  6.  0. 10.  2.
  6.  2.  0.  2.  8.  2.  4.  2.  2.  2.  6.  4.  4.  4.  6.  6.  2.  0.
  0.]
GO Right rates: [ 6.  4.  8.  2. 12.  2.  6.  2.  4.  0.  0.  2.  4.  6.  4.  8.  2.  6.
  0.  8.  4.  4.  0.  2.  0.  2. 10.  2. 12.  0.  0.  0.  2.  6.  6.  8.
  2.  0.  0.  2.  6.  0. 14.  0.  4. 10.  0.  8.  4.  2.  6.  2.  2.  6.
  6.  2.  6.  2.  6.  2.  6.  6.  2.  2.  4.  4.  2.  2.  2.  0.  0.  6.
  4.  2.  0. 12. 10.  2.  2.  0.  2.  4.  2.  0.  0.  6.  0.  4.  4.  2.
  6. 10.  2.  0.  8.  6.  0.  0.  2.  8.  0.  0.  2.  2.  4.  4.  0.  6.
  4.  2.  4.  0.  8.  2. 10.  4.  2.  8.  2.  6.  0.  2.  0.  4.  0.  8.]
ANOVA Results

In [29]:
# cell = session.get_cell(session.cell_ids[i])
cell = session.get_cell(9876)
print(f"i = {i}")
i += 1
rasters = cell.plot_raster_by_type_direction(
    epok=epok, alignment_point=align, 
)
rasters
psths = cell.plot_psth_by_type_direction(
    epok = epok,
    bin_size = 1,
    alignment_point = 'stop_cue',
    # alignment_point = align,
    separate_ssd = False,
    smooth = True,
    smooth_ker_size = 25,
    delta = False,
    normalize_bins = False
)


(
    # (psths[0][type] * psths[180][type]) + \
    (psths[0]['CONT'] * psths[180]['CONT']) + \
    rasters[0][type] + \
    rasters[180][type]
).cols(1)

i = 22


ValueError: stop_cue is NaN for this trial